<a href="https://colab.research.google.com/github/jabri62018/Jabri_lab/blob/Jabri_lab/Zx_Darkenergy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:

# file= Zx_Darkenergy.ipynb
# Author= Eng. Abdulla Al-Jabri
# Zero input: Dark energy from Z''' term

import mpmath as mp
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import zipfile
from IPython.display import display, HTML

print("="*60)
print("PROBLEM: Get Dark Energy density from Zx zeros")
print("Key: DE comes from Z''' in formula C_calc = 0.5 g^2 Re[z'''/z]")
print("Target: DE = 6.9e-27 kg/m3")
print("="*60)

mp.mp.dps = 80
xp = 21.0

def Zx(t):
    x = mp.mpc('0.5', str(t))
    return mp.exp(-x/xp) * mp.exp(-5*mp.log(x)) * mp.log(x) * mp.sin(2*mp.pi/x)

def find_zeros(n=30):
    mp.mp.dps = 60
    t_vals = np.arange(14.0, 200, 0.02)
    f_vals = [float(mp.im(Zx(t))) for t in t_vals]
    brackets = []
    for j in range(len(f_vals)-1):
        if f_vals[j] * f_vals[j+1] < 0:
            brackets.append((t_vals[j], t_vals[j+1]))
    mp.mp.dps = 80
    zeros = []
    for t_min, t_max in brackets[:n]:
        r = mp.findroot(lambda tt: mp.im(Zx(tt)), (t_min, t_max), tol=mp.mpf('1e-65'))
        zeros.append(float(r))
    return zeros

zeros = find_zeros(30)

CONSTANTS = {
    'DE': 6.9e-27, 't_H': 4.35e17, 'H0': 67.4, 'h': 6.62607015e-34,
}

def calc_C_and_Z3(gamma):
    h = mp.mpf('1e-15')
    t = mp.mpf(gamma)
    z = Zx(t)
    zppp = (Zx(t+2*h) - 2*Zx(t+h) + 2*Zx(t-h) - Zx(t-2*h)) / (2*h)
    C = float(0.5 * gamma**2 * mp.re(zppp / z))
    Z3_real = float(mp.re(zppp))
    return C, Z3_real

rows = []
for i, g in enumerate(zeros, 1):
    C, Z3 = calc_C_and_Z3(g)
    logC = np.log10(abs(C) + 1e-300)
    diffs = {k: abs(logC - np.log10(abs(v) + 1e-300)) for k,v in CONSTANTS.items()}
    matched = min(diffs, key=diffs.get)
    rows.append({'Root': i, 'gamma': g, 'C_calc': C, 'Z3_real': Z3,
                 'Matched': matched, 'Value': CONSTANTS[matched], 'Log Diff': diffs[matched]})

df = pd.DataFrame(rows)
df.to_csv('Zx_Darkenergy_match.csv', index=False, float_format='%.15e')

plt.figure(figsize=(11,6))
plt.loglog(df['C_calc'], df['gamma'], 'o-', markersize=6, label='Zx roots')
de_rows = df[df['Matched'] == 'DE']
if not de_rows.empty:
    plt.scatter(de_rows['C_calc'], de_rows['gamma'], s=120, c='red', zorder=5, label='DE match')
    for _, r in de_rows.iterrows():
        plt.annotate(f"DE LogDiff={r['Log Diff']:.2e}", (r['C_calc'], r['gamma']),
                     fontsize=10, weight='bold', ha='right')
plt.axvline(CONSTANTS['DE'], color='red', linestyle='--', alpha=0.6, label='DE target')
plt.xlabel('C_calc')
plt.ylabel('gamma zero')
plt.title("Zx Dark Energy: Z''' to DE")
plt.legend()
plt.grid(True, which='both', alpha=0.3)
plt.savefig('Zx_Darkenergy_plot.png', dpi=300)
plt.close()

print("\n=== TABLE: DE matches ===")
de_table = df[df['Matched'] == 'DE'][['Root','gamma','C_calc','Z3_real','Log Diff']]
if de_table.empty:
    print("No DE match in first 30 roots. Try n=50.")
    display(df[['Root','gamma','C_calc','Matched','Log Diff']].head(15))
else:
    display(de_table)

display(HTML('<img src="Zx_Darkenergy_plot.png" width="900">'))
print(f"\nTotal roots: {len(df)} | DE matches: {len(de_rows)}")
if not de_rows.empty:
    best = de_rows.loc[de_rows['Log Diff'].idxmin()]
    print(f"Best DE match: Root {int(best['Root'])} | Log Diff = {best['Log Diff']:.2e}")

with zipfile.ZipFile('Zx_Darkenergy_results.zip', 'w') as zipf:
    zipf.write('Zx_Darkenergy_match.csv')
    zipf.write('Zx_Darkenergy_plot.png')
from google.colab import files
files.download('Zx_Darkenergy_results.zip')
print("\nDone. Files downloaded.")

PROBLEM: Get Dark Energy density from Zx zeros
Key: DE comes from Z''' in formula C_calc = 0.5 g^2 Re[z'''/z]
Target: DE = 6.9e-27 kg/m3

=== TABLE: DE matches ===


,Root,gamma,C_calc,Z3_real,Log Diff
0,1,15.053925,-9.367783e-30,1.393814e-37,2.867212



Total roots: 3 | DE matches: 1
Best DE match: Root 1 | Log Diff = 2.87e+00


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Done. Files downloaded.
